In [67]:
import pandas as pd
import numpy as np
import os

In [68]:
#Adding functions to read and write the correct dtypes for the csv files
# Source - https://stackoverflow.com/a/50051542
# Posted by Aaron Brock, modified by community. See post 'Timeline' for change history
# Retrieved 2026-02-26, License - CC BY-SA 3.0

def to_csv(df, path):
    # Prepend dtypes to the top of df (from https://stackoverflow.com/a/43408736/7607701)
    df.loc[-1] = df.dtypes
    df.index = df.index + 1
    df.sort_index(inplace=True)
    # Then save it to a csv
    df.to_csv(path, index=False)

def read_csv(path):
    # Read types first line of csv
    dtypes = {key:value for (key,value) in pd.read_csv(path,
              nrows=1).iloc[0].to_dict().items() if 'date' not in value}

    parse_dates = [key for (key,value) in pd.read_csv(path,
                   nrows=1).iloc[0].to_dict().items() if 'date' in value]
    # Read the rest of the lines with the types from above
    return pd.read_csv(path, dtype=dtypes, parse_dates=parse_dates, skiprows=[1])

In [69]:
# Dictionary to store the desired dtypes for each column
# The datetime objects will have to be dealt with seperately, so we add those to the list of columns below

# First I will list all of the columns that appear in the data/raw_data/schedule_data with their desired dtypes
# Commenting out the ones that are going to be dropped so that we can just avoid reading them in
schedule_dtypes ={
    #"Unnamed: 0": "int64",
    #"Time": "string",
    "Vehicle": "float64",
    "Gap": "object", # I would really like for this to be a string but this caused some issues later so I will try to read it in as an object instead or just not specify the dtype?
    "Headway": "object",
    "Schedule": "object",
    "Destination": "object",
    #"day": "datetime64[us]",  # deal with date type seperately
    "day of the week": "int64",
    "stop": "string",
    #"Riders after stop": "float64"
}

# List all columns from dictionary keys plus any datetime columns
schedule_cols = ["day"] + list(schedule_dtypes.keys())

In [70]:
# Next read the data from each csv with the desired columns
schedule_data_dir = '../data/raw_data/schedule_data/'
year_dir = [x for x in os.listdir(schedule_data_dir) if x[:3] == '202' and x[-1] in ['5','6']]
print(year_dir)

['2025', '2026']


In [71]:
dataframes = []
for year in year_dir:
    data_files = os.listdir(os.path.join(schedule_data_dir, year))
    if 'with_stops' in data_files:
        csv_files = [file for file in os.listdir(os.path.join(schedule_data_dir,year,'with_stops')) if file[-4:] == '.csv']
        dataframes += [pd.read_csv(os.path.join(schedule_data_dir,year,'with_stops',file), usecols=schedule_cols) for file in csv_files]
    else:
        csv_files = [file for file in os.listdir(os.path.join(schedule_data_dir,year)) if file[-4:] == '.csv']
        dataframes += [pd.read_csv(os.path.join(schedule_data_dir,year,file), usecols=schedule_cols) for file in csv_files]

In [72]:
def find_datetime(day: str | float, time: str | float) -> pd.Timestamp | float:
    if type(day) != str or type(time) != str:
        return np.nan
    if time[0] == ' ':
        time = time[1:]
    return pd.Timestamp(f'{day} {time}')

In [73]:
for df in dataframes:
    df['Time'] = df.Schedule.str[-10:]
    df['scheduled time'] = df.apply(lambda x: find_datetime(day=x['day'], time=x['Time']), axis=1)
    df.drop(columns=['day', 'Time', 'day of the week'], inplace=True)

KeyboardInterrupt: 

In [ ]:
def east_or_west(dest: str| float) -> str| float:
    if type(dest) != str:
        return np.nan
    dest = dest.lower()
    if 'east' in dest:
        return 'E'
    else:
        return 'W'


In [ ]:
for df in dataframes:
    df['EB/WB'] = df['Destination'].apply(east_or_west)
    df.drop('Destination', axis=1, inplace=True)

In [ ]:
def min_delay(schedule: str|float) -> int|float:
    if type(schedule) != str:
        return np.nan

    schedule = schedule.lower()
    min_marker = schedule.find(':')
    hour_marker = schedule[:min_marker].rfind(' ')
    if hour_marker == -1:
        hour_marker = 0

    minute = (int(schedule[hour_marker:min_marker])
              +int(schedule[min_marker+1:min_marker+3])/60)

    if 'ahead' in schedule:
        return -minute
    elif 'behind' in schedule:
        return minute
    else:
        return 0

for df in dataframes:
    df['min delay'] = df['Schedule'].apply(min_delay)
    df.drop('Schedule', axis=1, inplace=True)

In [ ]:
cleaned_df = pd.concat(dataframes, ignore_index=True)


In [ ]:
cleaned_df.head()

In [ ]:
# We drop the instances where the min delay is over 2 hours, since that is most
# likely a cancellation rather than actual data.
cleaned_df = cleaned_df[cleaned_df['min delay'] < 120]

In [ ]:
# Here we prune the data frame from segments where there is no gap info.
# First let's make sure that the Gap column is in time format.
# Turning the gap into a timestamp is problematic because it is sometimes negative.
# As well, sometimes the gap is sometimes a few hours (so it is of the format hh:mm:ss).
# Therefore, I elect to turn it into a float, where the units are in seconds.
def gap_to_seconds(gap: str|float) -> float:
    if type(gap) != str:
        return np.nan

    if gap.find('-') == -1:
        is_negative = False
    else:
        gap = gap[1:]
        is_negative = True

    if gap.count(':') > 1:
        hours = int(gap[:-6])
        minutes = int(gap[-5:-3])
        seconds = int(gap[-2:])
    else:
        hours = 0
        minutes = int(gap[:-3])
        seconds = int(gap[-2:])

    total_seconds = (hours*60 + minutes)*60 + seconds
    if is_negative:
        return -total_seconds
    else:
        return total_seconds



cleaned_df.Gap = cleaned_df.Gap.apply(gap_to_seconds)

n= 25
gap_na = cleaned_df.Gap.isna()
# For each index, this gives the largest size of a sequence of NaNs containing
# the index if the index is a NaN, and if the index is not a NaN, it gives the
# largest size of a sequence of non-NaNs.
s= gap_na.groupby(gap_na.diff().ne(0).cumsum()).transform('count')
# We only keep the data where the sequence of gaps of NaNs is smaller than n.
# If the Gap index is NaN, then the component is at most n.
cleaned_df = cleaned_df.loc[~(gap_na)|(s<=n)]

In [ ]:
# I am going to elect to drop the times where there is no scheduling info
cleaned_df.dropna(axis=0, subset=['scheduled time'], inplace=True)

In [ ]:
# Finally, we time order cleaned_df.
cleaned_df.sort_values(by=['scheduled time'], ignore_index=True, inplace=True)

In [ ]:
cleaned_df.head()

This version of the data set is only going to look at delays at 10 minute intervals without the summary data included.

In [ ]:
cleaned_df['scheduled time'] = cleaned_df['scheduled time'].apply(pd.to_datetime)

In [ ]:
first_date = cleaned_df['scheduled time'].min()
first_date = pd.Timestamp(year=first_date.year, month=first_date.month, day=first_date.day)
last_date = cleaned_df['scheduled time'].max()
last_date = pd.Timestamp(year=last_date.year, month=last_date.month, day=last_date.day)
day_delta = (last_date - first_date).days
minute_delta = (day_delta+1)*24*60//15 # fifteen minute intervals from first date to last date.
print(day_delta)
hour_delta = (day_delta+1)*24

In [ ]:
stops = cleaned_df.stop.unique()
main_df = dict()
time_list = []
for stop in stops:
    for dir in ['E', 'W']:
        main_df[stop,dir] = pd.DataFrame({
            'stop': [stop for _ in range(hour_delta)],
            'EB/WB' : [dir for _ in range(hour_delta)],
            'time': [first_date + pd.Timedelta(hours=k) for k in range(hour_delta)], #Denotes the starting period of a time frame
        })

In [ ]:
def count_bunch(df: pd.DataFrame) -> int:
    return len(df.Gap[df.Gap <= 120])

def count_gap(df: pd.DataFrame) -> int:
    return len(df.Gap[df.Gap > 19*60])

In [ ]:
number_bunch = dict()
number_gap = dict()
dt_short = dict()
dt_long = dict()
for dir in ['E', 'W']:
    for stop in stops:
        number_bunch[dir,stop] = dict()
        number_gap[dir,stop] = dict()
        dt_short[dir,stop] = dict()
        dt_long[dir,stop] = dict()

filtered_cleaned = dict()

def write_df(df, time):
    start_time = first_date + pd.Timedelta(hours=time)
    end_time = start_time + pd.Timedelta(hours=1)
    return df[(df['scheduled time'] >= start_time) & (df['scheduled time'] < end_time)]

In [ ]:
from joblib import Parallel, delayed

for dir in ['E', 'W']:
    for stop in stops:
        filter_df = cleaned_df[(cleaned_df['stop'] == stop)&(cleaned_df['EB/WB'] == dir)]
        temp_list = Parallel(n_jobs=10)(delayed(write_df)(filter_df, time) for time in range(hour_delta))
        for time in range(hour_delta):
            filtered_cleaned[dir, stop, time] = temp_list[time]

In [ ]:
# Sanity check
display(filtered_cleaned['E', 'St George St', 40])

In [ ]:
def gather_data(dir, stop, time, dt) -> None:
    start_time = first_date + pd.Timedelta(hours=time)
    filtered_main = main_df[stop,dir][main_df[stop,dir]['time'] == start_time]

    filtered_df = filtered_cleaned[dir, stop, time]

    for row in filtered_main.index:
        number_bunch[dir,stop][row]=count_bunch(filtered_df)
        number_gap[dir,stop][row]=count_gap(filtered_df)

        if dt == 'short':
            dt_short[dir,stop][row]=sum(filtered_df[filtered_df['min delay'].between(5,19, inclusive='left')]['min delay'])
        else:
            dt_long[dir,stop][row]=sum(filtered_df[filtered_df['min delay'] >= 19]['min delay'])


In [ ]:
for dir in ['E', 'W']:
    for stop in stops:
        for dt in ['short', 'long']:
            for time in range(hour_delta):
                gather_data(dir, stop, time, dt)
            print(f'Done: {dir}, {stop}, {dt}')

In [ ]:
for dir in ['E','W']:
    for stop in stops:
        main_df[stop,dir]['bunch'] = [number_bunch[dir, stop][key] for key in sorted(number_bunch[dir,stop])]
        main_df[stop,dir]['gap'] = [number_gap[dir, stop][key] for key in sorted(number_gap[dir,stop])]
        main_df[stop,dir]['dt short'] = [dt_short[dir, stop][key] for key in sorted(dt_short[dir,stop])]
        main_df[stop,dir]['dt long'] = [dt_long[dir, stop][key] for key in sorted(dt_long[dir,stop])]

In [ ]:
display(main_df['St George St', 'E'])

In [ ]:
main_df = pd.concat(list(main_df.values()), ignore_index=True)

In [ ]:
main_df.head()

In [ ]:
def split_dataframe_into_chunks(df: pd.DataFrame, chunk_size: int):
    return [df[k:k+chunk_size] for k in range(0, len(df), chunk_size)]

df_split = split_dataframe_into_chunks(main_df, 300000)

In [ ]:
for idx, df in enumerate(df_split):
    to_csv(df,f'../data/schedule_data/processed_data/by_hour/schedule_data_with_stops_{idx}.csv')